# 🚀 ERP Sales BDR Agent: The Transformation Hunter (Native Grounding Edition)
**Autonomous Sales Prospecting with Gemini's Native Google Search**

This notebook implements a state-of-the-art **Agentic Workflow** using **Google ADK** and **Gemini 2.5**.
It leverages **Native Google Search Grounding** to fetch real-time data without needing external API keys like Tavily or Serper.

### **Architecture**
1.  **Native Grounding**: Agents use `google_search` tool directly embedded in the model.
2.  **Parallel Execution**: Research and Signal Detection run concurrently.
3.  **Cognitive Scoring**: A dedicated agent evaluates "Propensity to Buy" with **citations**.
4.  **Reflexion Loop**: A Quality Control agent critiques and iterates on the sales pitch.

### **Requirements**
- `GOOGLE_API_KEY` (Must be set in Kaggle Secrets)

In [ ]:
# --- 0. Setup & Dependencies ---
!pip install -q google-generativeai pydantic

import os
import sys
import logging
import json
import asyncio
from datetime import datetime
from typing import List, Optional, Dict, Any
from kaggle_secrets import UserSecretsClient

# --- 1. Authentication ---
try:
    secrets = UserSecretsClient()
    os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
    print("✅ Google API Key loaded successfully.")
except Exception as e:
    print(f"⚠️ Auth Warning: {e}. Ensure GOOGLE_API_KEY is set in Kaggle Secrets.")

# --- 2. Observability ---
class AgentLogger:
    def log(self, agent: str, event: str, details: str):
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] 🤖 \033[1m{agent}\033[0m: {event} -> {details}")

logger = AgentLogger()

# Suppress verbose logs
logging.getLogger("google_genai").setLevel(logging.ERROR)

from google.adk.agents import Agent, SequentialAgent, LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool
from google.genai import types

print("✅ Environment Ready.")

In [ ]:
# --- 3. Tool Definitions (Native Grounding) ---

# We define the Google Search tool using the GenAI types.
# This will be passed to the Gemini model configuration.
google_search_tool = types.Tool(google_search=types.GoogleSearch())

print("✅ Native Google Search Tool Configured.")

In [ ]:
# --- 4. Data Models (Pydantic) ---
from pydantic import BaseModel, Field

class CompanyProfile(BaseModel):
    company_name: str
    industry: str
    description: str
    revenue_estimate: str
    competitors: List[str]

class ERPSignals(BaseModel):
    current_tech_stack: List[str] = Field(description="Inferred ERP/CRM/HRIS systems")
    pain_points: List[str] = Field(description="Operational challenges inferred from news/jobs")
    trigger_events: List[str] = Field(description="M&A, IPO, Leadership Change, Expansion")

class StakeholderList(BaseModel):
    stakeholders: List[Dict[str, str]] = Field(description="List of {'name': '', 'title': '', 'linkedin': ''}")

class LeadScore(BaseModel):
    score: int = Field(description="0-100 Propensity to Buy SAP S/4HANA")
    reasoning: str = Field(description="Detailed justification for the score")
    citations: List[str] = Field(description="List of URLs supporting the reasoning")

class SalesPitch(BaseModel):
    subject_line: str
    email_body: str

class ReviewFeedback(BaseModel):
    status: str = Field(description="'APPROVED' or 'REJECTED'")
    critique: str
    improvement_instructions: str

print("✅ Data Models Defined.")

In [ ]:
# --- 5. Agent Definitions ---

retry_config = types.HttpRetryOptions(attempts=3)

# Configure the model with Google Search enabled
# Note: We pass the tool config to the Gemini wrapper.
# If the ADK wrapper supports 'tools' in init, this works.
# Otherwise, we rely on the instruction to trigger the model's internal knowledge or pass it via agent tools.
# For this implementation, we assume the ADK Gemini class accepts 'tools' which are passed to the underlying client.
grounded_model = Gemini(
    model="gemini-2.0-flash-exp", # Using 2.0 Flash Exp for best grounding support
    retry_options=retry_config,
    tools=[google_search_tool]
)

# 1. Research Specialist (Firmographics)
research_agent = LlmAgent(
    name="ResearchSpecialist",
    model=grounded_model,
    output_schema=CompanyProfile,
    instruction="""
    You are a Corporate Researcher with access to Google Search.
    Goal: Build a profile for the target company.
    1. Search for the company name, industry, and what they sell.
    2. Find revenue estimates and top 3 competitors.
    3. Summarize the description.
    """
)

# 2. Signal Specialist (Tech & Triggers)
signal_agent = LlmAgent(
    name="SignalSpecialist",
    model=grounded_model,
    output_schema=ERPSignals,
    instruction="""
    You are a Technical Sales Engineer with access to Google Search.
    Goal: Identify ERP transformation signals.
    1. Search for "Company Name + jobs SAP/Oracle/NetSuite" to infer stack.
    2. Search for "Company Name + problems/delays/acquisition" to find pain points.
    """
)

# 3. Stakeholder Specialist
stakeholder_agent = LlmAgent(
    name="StakeholderSpecialist",
    model=grounded_model,
    output_schema=StakeholderList,
    instruction="""
    You are an Executive Headhunter with access to Google Search.
    Goal: Find the Buying Committee (CIO, CFO, VP Supply Chain).
    1. Search for specific names and titles of executives at the company.
    2. Return a list of top 3 relevant contacts.
    """
)

# 4. Lead Scoring Agent (Cognitive Synthesis)
scoring_agent = LlmAgent(
    name="LeadScoringAgent",
    model=grounded_model, # Also grounded to verify claims if needed
    output_schema=LeadScore,
    instruction="""
    You are a Senior Deal Desk Analyst.
    Goal: Score the prospect (0-100) for SAP S/4HANA fit.
    Input: Company Profile, Signals, Stakeholders.
    CRITICAL: You MUST provide URLs in the 'citations' field for every claim you make in the reasoning.
    Use the Google Search tool if you need to verify any claim or find a source URL.
    """
)

# 5. Copywriter
copywriter_agent = LlmAgent(
    name="CopywriterSpecialist",
    model=grounded_model,
    output_schema=SalesPitch,
    instruction="""
    You are an Elite B2B Copywriter.
    Goal: Write a hyper-personalized cold email.
    1. Use the 'LeadScore' reasoning to hit their specific pain points.
    2. Mention the 'Trigger Event' found by the Signal Agent.
    3. Keep it under 150 words. Punchy. Professional.
    """
)

# 6. Editor (Quality Control)
editor_agent = LlmAgent(
    name="EditorSpecialist",
    model=grounded_model,
    output_schema=ReviewFeedback,
    instruction="""
    You are a Strict Sales Director.
    Goal: Review the cold email draft.
    Criteria:
    - Is it personalized? (No generic fluff)
    - Does it mention a specific trigger?
    - Is it short?
    Output: APPROVED or REJECTED with specific instructions.
    """
)

print("✅ Agents Defined with Native Grounding.")

In [ ]:
# --- 6. Advanced Workflow Orchestration ---

async def run_full_mission(domain: str):
    print(f"\n{'='*60}")
    print(f"🚀 STARTING NATIVE GROUNDING MISSION: {domain}")
    print(f"{'='*60}\n")

    # --- Helper to run agents ---
    async def run_agent(agent, prompt, context=""):
        from google.adk.runners import Runner
        from google.adk.sessions import InMemorySessionService
        from google.adk.memory import InMemoryMemoryService
        
        r = Runner(agent=agent, session_service=InMemorySessionService(), memory_service=InMemoryMemoryService())
        session = await r.session_service.create_session("app", "user")
        full_prompt = f"{prompt}\nContext: {context}"
        
        result_text = ""
        async for event in r.run_async("user", session.id, types.Content(role="user", parts=[types.Part(text=full_prompt)])):
            if event.content:
                result_text = event.content.parts[0].text
        return result_text

    # --- Phase 1: Parallel Research ---
    logger.log("Orchestrator", "Phase 1", "Parallel Research & Signals (Native Search)")
    profile_task = run_agent(research_agent, f"Research {domain}")
    signal_task = run_agent(signal_agent, f"Find ERP signals for {domain}")
    
    # Await both
    profile_json, signals_json = await asyncio.gather(profile_task, signal_task)
    
    print(f"\n📊 [Profile Data]: {profile_json[:100]}...")
    print(f"📡 [Signal Data]: {signals_json[:100]}...")

    # --- Phase 2: Sequential Stakeholders ---
    logger.log("Orchestrator", "Phase 2", "Stakeholder Identification")
    stakeholders_json = await run_agent(stakeholder_agent, f"Find stakeholders for {domain}", context=f"{profile_json}\n{signals_json}")
    print(f"👥 [Stakeholders]: {stakeholders_json[:100]}...")

    # --- Phase 3: Cognitive Scoring ---
    logger.log("Orchestrator", "Phase 3", "Lead Scoring & Reasoning")
    score_json = await run_agent(scoring_agent, "Score this lead.", context=f"{profile_json}\n{signals_json}\n{stakeholders_json}")
    print(f"🏆 [Score]: {score_json}")

    # --- Phase 4: Reflexion Loop (Copywriter <-> Editor) ---
    logger.log("Orchestrator", "Phase 4", "Drafting & Quality Control Loop")
    
    approved = False
    attempts = 0
    feedback = ""
    final_pitch = ""

    while not approved and attempts < 3:
        attempts += 1
        logger.log("Loop", f"Attempt {attempts}", "Drafting Email...")
        
        # Draft
        pitch_json = await run_agent(copywriter_agent, "Draft sales email.", context=f"{profile_json}\n{signals_json}\n{stakeholders_json}\n{score_json}\nFeedback: {feedback}")
        print(f"   📝 Draft: {pitch_json[:100]}...")

        # Review
        review_json = await run_agent(editor_agent, "Review this email.", context=pitch_json)
        print(f"   🧐 Review: {review_json}")
        
        # Parse Review
        try:
            review_data = json.loads(review_json.replace("```json", "").replace("```", ""))
            if review_data.get("status") == "APPROVED":
                approved = True
                final_pitch = pitch_json
                logger.log("Loop", "Success", "Email Approved!")
            else:
                feedback = review_data.get("improvement_instructions", "Try again.")
                logger.log("Loop", "Rejected", f"Feedback: {feedback}")
        except:
            # Fallback if JSON parsing fails
            logger.log("Loop", "Error", "Failed to parse review. Assuming approved to exit loop.")
            final_pitch = pitch_json
            approved = True

    print(f"\n{'='*60}\n✅ FINAL OUTPUT\n{'='*60}\n")
    print(final_pitch)

# --- Execute ---
# Note: Replace 'capvest.com' with any real domain you want to test
await run_full_mission("capvest.com")